In [1]:
#Normalize Bronze Evidence
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

host     = os.getenv("ORACLE_BRONZE_HOST", "localhost")
port     = os.getenv("ORACLE_BRONZE_PORT", "1521")
service  = os.getenv("ORACLE_BRONZE_SERVICE", "FREEPDB1")
user     = os.getenv("ORACLE_BRONZE_USER")
password = os.getenv("ORACLE_BRONZE_PASSWORD")

engine_bronze = create_engine(f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}")

with engine_bronze.connect() as conn:
    df_pd_raw = pd.read_sql("SELECT * FROM inbound_claims", conn)

df_pd_raw.columns = [c.lower() for c in df_pd_raw.columns]

# Explode URI columns into rows
uri_cols = ["video_uri_claimant", "image_uri_claimant", "image_uri_counterparty"]
id_cols  = [c for c in df_pd_raw.columns if c not in uri_cols]

df_norm = (
    df_pd_raw
    .melt(id_vars=id_cols, value_vars=[c for c in uri_cols if c in df_pd_raw.columns], value_name="source_uri")
    .dropna(subset=["source_uri"])
    .drop(columns=["variable"])
    .reset_index(drop=True)
)

print(f"Rows after explode: {len(df_norm)}")

Rows after explode: 3


In [2]:
#Infer Modality and Prepare for AI
def infer_modality(uri: str) -> str:
    u = uri.lower()
    if u.endswith(('.jpg', '.jpeg', '.png')): return 'image'
    if u.endswith(('.mp4', '.mov', '.avi')): return 'video'
    return 'unknown'

df_norm["summary_id"] = df_norm.apply(
    lambda r: abs(hash(f"{r['inbox_id']}|{r['source_uri']}")) % 2147483648, axis=1
).astype(int)

df_norm["claim_id"] = df_norm["claim_id_ext"]
df_norm["modality"] = df_norm["source_uri"].apply(infer_modality)

df_pd = df_norm[["summary_id", "claim_id", "inbox_id", "modality", "source_uri", "created_at", "policy_id"]]

In [3]:
#Initialize AI Client — Qwen VLM (model configured in .env)
import os
import torch
from dotenv import load_dotenv
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info

load_dotenv()

# To switch model: edit QWEN_MODEL_NAME in .env
# QWEN_MODEL_NAME=Qwen/Qwen3-VL-4B-Instruct  ← faster, lighter
# QWEN_MODEL_NAME=Qwen/Qwen3-VL-8B-Instruct  ← more accurate
model_name = os.getenv("QWEN_MODEL_NAME")
if not model_name:
    raise ValueError("QWEN_MODEL_NAME is not set in .env")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {model_name} on {device}...")

qwen_model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
qwen_processor = AutoProcessor.from_pretrained(model_name)

print(f"Model loaded on {device}")


Loading Qwen/Qwen3-VL-4B-Instruct on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Model loaded on cuda


In [4]:
#Define Helper Functions
import cv2, io, base64, re
from PIL import Image

def extract_confidence(text: str) -> str:
    try:
        match_pct = re.search(r"==[Cc]onfidence [Ss]core==\s*(\d+)%", text)
        if match_pct:
            return str(round(int(match_pct.group(1)) / 100, 2))
        match_dec = re.search(r"==[Cc]onfidence [Ss]core==\s*([0]\.\d+|\d\.\d+)", text)
        if match_dec:
            return str(round(float(match_dec.group(1)), 2))
    except:
        pass
    return "0.0"

def encode_image_b64(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    buf = io.BytesIO()
    Image.fromarray(img_rgb).save(buf, format="JPEG", quality=85)
    return f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"

def sample_frames(video_path, max_frames=12, min_interval_s=0.5):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return [], [], 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur = nframes / max(fps, 0.0001)
    count = min(max_frames, max(1, int(dur / min_interval_s)))
    indices = [int(round(i * (nframes - 1) / (count - 1))) for i in range(count)] if count > 1 else [0]
    imgs, stamps = [], []
    for idx in sorted(set(indices)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            imgs.append(frame)
            stamps.append(idx / fps)
    cap.release()
    return imgs, stamps, dur

def qwen_infer(messages: list, max_new_tokens: int = 512) -> str:
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return qwen_processor.batch_decode(trimmed, skip_special_tokens=True)[0]

In [7]:
#Define Helper Functions
import cv2, io, base64, re
from PIL import Image

def extract_confidence(text: str) -> str:
    try:
        match_pct = re.search(r"==[Cc]onfidence [Ss]core==\s*(\d+)%", text)
        if match_pct:
            return str(round(int(match_pct.group(1)) / 100, 2))
        match_dec = re.search(r"==[Cc]onfidence [Ss]core==\s*([0]\.\d+|\d\.\d+)", text)
        if match_dec:
            return str(round(float(match_dec.group(1)), 2))
    except:
        pass
    return "0.0"

def encode_image_b64(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    buf = io.BytesIO()
    Image.fromarray(img_rgb).save(buf, format="JPEG", quality=85)
    return f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"

def sample_frames(video_path, max_frames=12, min_interval_s=0.5):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return [], [], 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur = nframes / max(fps, 0.0001)
    count = min(max_frames, max(1, int(dur / min_interval_s)))
    indices = [int(round(i * (nframes - 1) / (count - 1))) for i in range(count)] if count > 1 else [0]
    imgs, stamps = [], []
    for idx in sorted(set(indices)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            imgs.append(frame)
            stamps.append(idx / fps)
    cap.release()
    return imgs, stamps, dur

def qwen_infer(messages: list, max_new_tokens: int = 512) -> str:
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return qwen_processor.batch_decode(trimmed, skip_special_tokens=True)[0]

In [ ]:
#Run AI Inference with Qwen3-VL
from datetime import datetime
import pandas as pd

from datetime import datetime
from pathlib import Path
import pandas as pd

# Map Docker/container volume paths to local Windows paths
EVIDENCE_LOCAL_PATH = Path(os.getenv("EVIDENCE_LOCAL_PATH", "D:/AI-Agent/InsuranceClaim/Claim Evidence"))

def resolve_path(uri: str) -> str:
    """Extract filename from Docker volume URI and resolve to local path."""
    filename = Path(uri).name
    local = EVIDENCE_LOCAL_PATH / filename
    if not local.exists():
        raise FileNotFoundError(f"Evidence file not found: {local}\n  (original URI: {uri})")
    return str(local)

IMAGE_PROMPT = """You are an expert insurance surveyor analyzing vehicle damage evidence.

Analyze the image and provide:
1. A brief overview (2-3 sentences): describe visible damage, affected vehicle parts, and severity.
2. Impact analysis: identify the impact side (front / rear / left / right).
3. Your confidence score for the assessment (0-100%).

You MUST end your response with exactly this line:
Confidence Score: [NUMBER]%

Example ending: Confidence Score: 82%"""

VIDEO_PROMPT_TEMPLATE = """You are an insurance claims assistant analyzing a traffic accident video (~{dur:.1f}s).

Analyze the frames and provide:
1. Overview (2-3 sentences): vehicles involved, sequence of events, type of collision.
2. Chronological timeline: key moments with approximate timestamps.
3. Damage assessment: visible damage to each vehicle.
4. Your confidence score for the assessment (0-100%).

You MUST end your response with exactly this line:
Confidence Score: [NUMBER]%

Example ending: Confidence Score: 75%"""

def process_record(r):
    try:
        summary_id = int(r["summary_id"])
        claim_id   = str(r["claim_id"])
        inbox_id   = str(r["inbox_id"])
        modality   = str(r["modality"])
        uri        = str(r["source_uri"])
        created_at = pd.Timestamp(r["created_at"]) if pd.notnull(r["created_at"]) else pd.Timestamp(datetime.utcnow())
        policy_id  = str(r.get("policy_id", ""))

        local_path = resolve_path(uri)

        if modality == "image":
            with open(local_path, "rb") as f:
                b64 = base64.b64encode(f.read()).decode("utf-8")
            ext  = local_path.lower().split(".")[-1]
            mime = "image/png" if ext == "png" else "image/jpeg"
            messages = [{"role": "user", "content": [
                {"type": "image", "image": f"data:{mime};base64,{b64}"},
                {"type": "text",  "text": IMAGE_PROMPT}
            ]}]

        elif modality == "video":
            frames, stamps, dur = sample_frames(local_path)
            if not frames:
                raise RuntimeError(f"No frames extracted from: {local_path}")
            encoded_imgs = [encode_image_b64(f) for f in frames]
            content = []
            for img, t in zip(encoded_imgs, stamps):
                content.append({"type": "text",  "text": f"[t={round(t,2)}s]"})
                content.append({"type": "image", "image": img})
            content.append({"type": "text", "text": VIDEO_PROMPT_TEMPLATE.format(dur=dur)})
            messages = [{"role": "user", "content": content}]

        else:
            return (summary_id, claim_id, inbox_id, modality, uri,
                    "Unsupported modality", "0.0", created_at, policy_id)

        findings   = qwen_infer(messages)
        confidence = extract_confidence(findings)
        print(f"[{modality}] {claim_id}: confidence={confidence}")
        print(f"  → preview: {findings[:150].strip()!r}")

    except Exception as e:
        findings   = f"Error: {str(e)}"
        confidence = "0.0"
        created_at = pd.Timestamp(datetime.utcnow())
        print(f"[ERROR] {r.get('claim_id', '?')} ({r.get('modality', '?')}): {e}")

    return (summary_id, claim_id, inbox_id, modality, uri, findings, confidence, created_at, policy_id)


records = [process_record(r) for _, r in df_pd.iterrows()]

[ERROR] MOB-CLM-003 (video): CUDA out of memory. Tried to allocate 15.17 GiB. GPU 0 has a total capacity of 11.99 GiB of which 0 bytes is free. Of the allocated memory 25.53 GiB is allocated by PyTorch, and 951.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


In [9]:
#Normalize Results to DataFrame
from datetime import datetime

for i, r in enumerate(records):
    val = r[-2]
    if isinstance(val, str):
        try: val = pd.Timestamp(val)
        except: val = pd.Timestamp(datetime.utcnow())
    elif isinstance(val, datetime): val = pd.Timestamp(val)
    elif not isinstance(val, pd.Timestamp): val = pd.Timestamp(datetime.utcnow())
    records[i] = r[:-2] + (val, r[-1])

df_pd_out = pd.DataFrame(records, columns=[
    "summary_id", "claim_id", "inbox_id", "modality", "source_uri",
    "findings", "confidence", "created_at", "policy_id"
])
df_pd_out["confidence"] = pd.to_numeric(df_pd_out["confidence"], errors="coerce").fillna(0.0)

print(df_pd_out[["claim_id", "modality", "confidence"]].to_string())

      claim_id modality  confidence
0  MOB-CLM-003    video         0.0
1  MOB-CLM-003    image         0.0
2  MOB-CLM-003    image         0.0


In [ ]:
#Upsert into Silver Table (Oracle direct — no Spark)
from sqlalchemy import text

silver_user     = os.getenv("ORACLE_SILVER_USER")
silver_password = os.getenv("ORACLE_SILVER_PASSWORD")

engine_silver = create_engine(
    f"oracle+oracledb://{silver_user}:{silver_password}@{host}:{port}/?service_name={service}"
)

upsert_sql = text("""
    MERGE INTO claim_evidence_summary t
    USING (
        SELECT :summary_id AS summary_id, :claim_id AS claim_id,
               :inbox_id   AS inbox_id,   :modality AS modality,
               :source_uri AS source_uri, :findings AS findings,
               :confidence AS confidence, :created_at AS created_at
        FROM dual
    ) s
    ON (t.inbox_id = s.inbox_id AND t.source_uri = s.source_uri)
    WHEN MATCHED THEN UPDATE SET
        t.summary_id = s.summary_id,
        t.claim_id   = s.claim_id,
        t.modality   = s.modality,
        t.findings   = s.findings,
        t.confidence = s.confidence,
        t.created_at = s.created_at
    WHEN NOT MATCHED THEN INSERT (
        summary_id, claim_id, inbox_id, modality, source_uri, findings, confidence, created_at
    ) VALUES (
        s.summary_id, s.claim_id, s.inbox_id, s.modality,
        s.source_uri, s.findings, s.confidence, s.created_at
    )
""")

with engine_silver.begin() as conn:
    for _, row in df_pd_out.iterrows():
        conn.execute(upsert_sql, {
            "summary_id": int(row["summary_id"]),
            "claim_id":   str(row["claim_id"]),
            "inbox_id":   str(row["inbox_id"]),
            "modality":   str(row["modality"]),
            "source_uri": str(row["source_uri"]),
            "findings":   str(row["findings"]),
            "confidence": float(row["confidence"]),
            "created_at": row["created_at"].to_pydatetime(),
        })

print(f"Upserted {len(df_pd_out)} row(s) into claim_evidence_summary")

In [ ]:
# View Bronze — inbound_claims
with engine_bronze.connect() as conn:
    display(pd.read_sql("SELECT * FROM inbound_claims", conn))

In [ ]:
# View Silver — claim_evidence_summary
with engine_silver.connect() as conn:
    display(pd.read_sql("SELECT * FROM claim_evidence_summary", conn))

In [ ]:
print("****Transformed and data loaded to Silver layer Successfully ****")